# Sentence Transformers — Semantic Embeddings for Real-World Search

## What Is This Notebook About?

Imagine you type "affordable pizza places nearby" into a search engine. A keyword search fails if a restaurant's description says "budget-friendly Italian cuisine close to you" — same meaning, zero matching words. **Sentence Transformers** solve this by converting text into *meaning vectors* so that "affordable pizza nearby" and "budget Italian food close to you" end up *near each other* in math-space.

This notebook teaches you **everything** about Sentence Transformers from scratch — no prior NLP knowledge required beyond basic Python.

---

## Why Should You Care?

| Real-World Problem | How Sentence Transformers Help |
|---|---|
| Customer support FAQ routing | Match user question → nearest FAQ answer |
| Job matching | Compare resume skills → job description semantically |
| Duplicate question detection | Stack Overflow / Quora dedup |
| Legal document search | Find similar clauses across contracts |
| Recommendation systems | "Customers who liked X" via text similarity |
| Plagiarism detection | Detect paraphrased content |

---

## Prerequisites

- Basic Python (lists, loops, functions)
- NumPy arrays (shape, dot product)
- Vague idea that neural networks exist

**You do NOT need to know**: transformers, BERT, attention mechanisms (we'll explain what you need)

---

## Table of Contents

1. [Setup & Installation](#1-setup)
2. [Word Embeddings vs Sentence Embeddings](#2-embeddings)
3. [How SBERT Works (Plain English)](#3-sbert-internals)
4. [Your First Sentence Embeddings](#4-first-embeddings)
5. [Cosine Similarity — The Distance Between Meanings](#5-cosine)
6. [Semantic Search from Scratch](#6-semantic-search)
7. [Clustering Text with K-Means](#7-clustering)
8. [Bi-Encoder vs Cross-Encoder](#8-bi-vs-cross)
9. [Fine-Tuning SBERT on Your Own Data](#9-finetuning)
10. [Mini Project — FAQ Semantic Search Engine](#10-mini-project)
11. [Common Pitfalls](#11-pitfalls)
12. [Interview Q&A](#12-interview)
13. [Resources](#13-resources)
14. [Summary](#14-summary)

---
## 1. Setup & Installation <a id='1-setup'></a>

In [ ]:
# Install dependencies (run once)
# !pip install sentence-transformers scikit-learn matplotlib numpy

import warnings
warnings.filterwarnings('ignore')

import numpy as np
import matplotlib.pyplot as plt
import matplotlib.cm as cm
from sklearn.cluster import KMeans
from sklearn.decomposition import PCA
from sklearn.metrics.pairwise import cosine_similarity
from collections import defaultdict
import time

try:
    from sentence_transformers import SentenceTransformer, util, InputExample, losses
    from sentence_transformers.evaluation import EmbeddingSimilarityEvaluator
    from torch.utils.data import DataLoader
    import torch
    SBERT_AVAILABLE = True
    print(f"sentence-transformers ready")
    print(f"PyTorch version: {torch.__version__}")
except ImportError:
    SBERT_AVAILABLE = False
    print("sentence-transformers not installed — pip install sentence-transformers")
    print("All demos will run in simulation mode.")

np.random.seed(42)
print("\nReady!")

---
## 2. Word Embeddings vs Sentence Embeddings <a id='2-embeddings'></a>

### The Library Analogy

**Word embeddings (Word2Vec, GloVe, FastText):**
- Like a dictionary — each word has one fixed definition card regardless of context
- "bank" always has the same vector, whether you mean river bank or bank account
- No understanding of full sentences

**Sentence Embeddings (SBERT):**
- Like a librarian who reads the whole paragraph and summarizes the *meaning* into one index card
- "I went to the bank to deposit money" vs "I sat by the river bank" → two different vectors
- Full sentence meaning captured in one fixed-size vector (384 or 768 numbers)

### The Key Problem SBERT Solves

Raw BERT gives great contextual embeddings, but comparing two sentences requires passing **both together** through BERT. With 10,000 sentences that's ~50 million forward passes = **65 hours on a GPU**.

SBERT encodes each sentence **independently** → comparison = dot product = **5 seconds** for the same 10,000 sentences.

In [ ]:
# ── Conceptual demo: word vs sentence embeddings ─────────────────────────────

# Word embeddings: same word always gets same vector (no context)
WORD_VECTORS = {
    'bank':    np.array([0.5, 0.3, -0.1]),
    'river':   np.array([0.6, 0.2, -0.2]),
    'deposit': np.array([0.1, 0.8,  0.3]),
    'money':   np.array([0.0, 0.9,  0.4]),
    'sat':     np.array([0.4, 0.1, -0.5]),
}

def cosine_sim(a, b):
    return np.dot(a, b) / (np.linalg.norm(a) * np.linalg.norm(b))

def naive_sentence_vec(words):
    """Mean pool of word vectors — how BOW-style models work."""
    vecs = [WORD_VECTORS[w] for w in words if w in WORD_VECTORS]
    return np.mean(vecs, axis=0)

s1_words = ['bank', 'deposit', 'money']
s2_words = ['bank', 'river', 'sat']

v1 = naive_sentence_vec(s1_words)
v2 = naive_sentence_vec(s2_words)

print("=== Word Embedding (Mean Pool) Approach ===")
print(f"Sentence 1: 'bank deposit money' → {v1.round(3)}")
print(f"Sentence 2: 'bank river sat'     → {v2.round(3)}")
print(f"Cosine similarity: {cosine_sim(v1, v2):.3f}")
print("  → These seem similar because 'bank' vector is shared!")
print("  → Word embeddings can't distinguish the two meanings.")
print()
print("=== SBERT Approach ===")
print("SBERT reads the WHOLE sentence in context:")
print("  'I went to the bank to deposit money' → finance meaning vector")
print("  'I sat by the river bank'             → geography meaning vector")
print("  → Low similarity despite sharing 'bank' — correct behavior!")

---
## 3. How SBERT Works (Plain English) <a id='3-sbert-internals'></a>

### The Siamese Twin Network

SBERT uses a **siamese network** — two identical neural networks (sharing weights) that each process one sentence independently:

```
Sentence A ──→ [BERT encoder] ──→ Mean Pool ──→ Vector A (384 dims)
                     ↕ (shared weights)
Sentence B ──→ [BERT encoder] ──→ Mean Pool ──→ Vector B (384 dims)
                                                       ↓
                              Cosine Similarity = semantic score
```

### How It's Trained

Trained on NLI (Natural Language Inference) + STS (Semantic Textual Similarity) datasets:
- **Entailment** (same meaning): push vectors close together
- **Contradiction** (opposite meaning): push vectors far apart

Loss: `TripletLoss = max(0, d(anchor, positive) - d(anchor, negative) + margin)`

### Mean Pooling (simplified)

```python
# BERT gives one vector per token → average them (ignore [PAD] tokens)
token_embeddings = bert_output.last_hidden_state  # [batch, seq_len, 768]
mask = attention_mask.unsqueeze(-1).expand(token_embeddings.size()).float()
sentence_embedding = torch.sum(token_embeddings * mask, dim=1)
sentence_embedding /= torch.clamp(mask.sum(dim=1), min=1e-9)
```

### Popular Models

| Model | Dims | Speed | Best For |
|---|---|---|---|
| `all-MiniLM-L6-v2` | 384 | Very fast | General purpose (start here) |
| `all-mpnet-base-v2` | 768 | Medium | Best quality general purpose |
| `multi-qa-MiniLM-L6-cos-v1` | 384 | Fast | Q&A / semantic search |
| `paraphrase-multilingual-MiniLM-L12-v2` | 384 | Medium | 50+ languages |

**Rule of thumb:** Start with `all-MiniLM-L6-v2`. Only switch if benchmarks show quality problems.

**Official docs:** https://www.sbert.net/docs/pretrained_models.html  
**Research paper:** https://arxiv.org/abs/1908.10084 (Reimers & Gurevych, 2019)

---
## 4. Your First Sentence Embeddings <a id='4-first-embeddings'></a>

In [ ]:
# ── Load model and encode sentences ─────────────────────────────────────────

sentences = [
    # Finance
    "I need to transfer money to my savings account.",
    "How do I invest in the stock market?",
    "What is the best credit card for cashback rewards?",
    # Food
    "What's a good recipe for chocolate cake?",
    "How do I make homemade pasta from scratch?",
    "What are the best spices for Indian cooking?",
    # Technology
    "How does machine learning work?",
    "What is the difference between RAM and storage?",
    "Explain how the internet works to a child.",
    # Health
    "What are the symptoms of vitamin D deficiency?",
    "How much water should I drink per day?",
    "What exercises help with lower back pain?",
]

if SBERT_AVAILABLE:
    print("Loading 'all-MiniLM-L6-v2' (first run downloads ~90 MB)...")
    model = SentenceTransformer('all-MiniLM-L6-v2')

    start = time.time()
    embeddings = model.encode(
        sentences,
        batch_size=32,
        show_progress_bar=True,
        normalize_embeddings=True   # L2 normalize → cosine sim = dot product
    )
    elapsed = time.time() - start

    print(f"\nEncoded {len(sentences)} sentences in {elapsed:.2f}s")
    print(f"Shape: {embeddings.shape}  ({embeddings.shape[0]} sentences × {embeddings.shape[1]} dims)")
    print(f"First 8 dims of sentence 0: {embeddings[0, :8].round(4)}")
else:
    # Simulation: synthetic embeddings with 4-cluster structure
    print("[SIMULATION] Generating synthetic cluster-structured embeddings")
    DIM = 384
    centers = np.random.randn(4, DIM)
    embeddings = []
    for c in range(4):
        for _ in range(3):
            e = centers[c] + np.random.randn(DIM) * 0.05
            e /= np.linalg.norm(e)
            embeddings.append(e)
    embeddings = np.array(embeddings)
    print(f"Simulated shape: {embeddings.shape}")

In [ ]:
# ── Visualize embeddings with PCA ────────────────────────────────────────────

pca = PCA(n_components=2, random_state=42)
reduced = pca.fit_transform(embeddings)

labels = ['Finance']*3 + ['Food']*3 + ['Technology']*3 + ['Health']*3
colors = {'Finance': '#e74c3c', 'Food': '#2ecc71',
          'Technology': '#3498db', 'Health': '#f39c12'}

fig, ax = plt.subplots(figsize=(11, 7))
plotted = set()
for i, (x, y) in enumerate(reduced):
    cat = labels[i]
    lbl = cat if cat not in plotted else ''
    ax.scatter(x, y, c=colors[cat], s=130, alpha=0.85, label=lbl, edgecolors='white')
    plotted.add(cat)
    ax.annotate(sentences[i][:32] + '...', (x, y), fontsize=7,
                xytext=(5, 4), textcoords='offset points')

ax.legend(fontsize=11)
pct = pca.explained_variance_ratio_.sum() * 100
ax.set_title(f'Sentence Embeddings in 2D (PCA) — {pct:.1f}% variance explained\n'
             'Same-topic sentences cluster together automatically!', fontsize=12)
ax.set_xlabel('PC1'); ax.set_ylabel('PC2')
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

print("KEY INSIGHT: No labels were given during encoding.")
print("SBERT discovered these topic groups purely from meaning.")

---
## 5. Cosine Similarity — The Distance Between Meanings <a id='5-cosine'></a>

### The Compass Analogy

Each embedding is a *direction* in 384-D space (like a compass direction). Cosine similarity measures the **angle** between two directions:
- Angle = 0° → same meaning → similarity **= 1.0**
- Angle = 90° → unrelated → similarity **= 0.0**
- Angle = 180° → opposite meaning → similarity **= −1.0**

Formula: `cos(θ) = (A · B) / (|A| × |B|)`

When embeddings are L2-normalized (unit length): `cos(θ) = A · B` — just a dot product, very fast!

In [ ]:
# ── Pairwise similarity heatmap ───────────────────────────────────────────────

sim_matrix = cosine_similarity(embeddings)

fig, ax = plt.subplots(figsize=(12, 10))
im = ax.imshow(sim_matrix, cmap='RdYlGn', vmin=-0.1, vmax=1.0)
plt.colorbar(im, ax=ax, shrink=0.8, label='Cosine Similarity')

short = [s[:35] + '...' for s in sentences]
ax.set_xticks(range(len(sentences))); ax.set_xticklabels(short, rotation=45, ha='right', fontsize=7)
ax.set_yticks(range(len(sentences))); ax.set_yticklabels(short, fontsize=7)

for i in range(len(sentences)):
    for j in range(len(sentences)):
        v = sim_matrix[i, j]
        c = 'white' if v > 0.7 or v < 0.1 else 'black'
        ax.text(j, i, f'{v:.2f}', ha='center', va='center', fontsize=6, color=c)

for b in [2.5, 5.5, 8.5]:
    ax.axhline(b, color='navy', lw=2)
    ax.axvline(b, color='navy', lw=2)

ax.set_title('Pairwise Cosine Similarity Matrix\n'
             'Green = similar meaning | Navy lines = topic cluster boundaries', fontsize=11)
plt.tight_layout()
plt.show()

# Manual similarity examples
print("\n=== Similarity Examples ===")
pairs = [
    (0, 1, "Finance vs Finance"),
    (3, 4, "Food vs Food"),
    (0, 6, "Finance vs Technology"),
    (0, 3, "Finance vs Food"),
]
for i, j, label in pairs:
    sim = float(np.dot(embeddings[i], embeddings[j]))
    bar = '█' * int(sim * 25)
    print(f"  {label:25s}: {sim:.3f}  {bar}")

---
## 6. Semantic Search from Scratch <a id='6-semantic-search'></a>

### The Recipe Book Analogy

Traditional search = scanning a recipe book index for the exact word "pasta"  
Semantic search = asking the librarian "I want Italian food" → gets pasta, risotto, bruschetta — no exact word match needed

**Steps:**
1. Encode all documents into vectors **once** (store them)
2. On query: encode query → dot product with all doc vectors
3. Return top-k highest scores

In [ ]:
# ── Corpus and encoding ───────────────────────────────────────────────────────

CORPUS = [
    # Python
    "Python list comprehensions allow creating lists with a concise syntax.",
    "How to handle exceptions in Python using try-except blocks.",
    "Python decorators wrap functions to add functionality without changing their code.",
    "Using virtual environments to manage Python project dependencies.",
    # Machine Learning
    "Overfitting occurs when a model memorizes training data but fails on new data.",
    "Cross-validation splits data multiple times to get reliable model performance estimates.",
    "Gradient descent is an optimization algorithm that minimizes the loss function.",
    "Feature engineering transforms raw data into inputs that machine learning models can use.",
    # Web Dev
    "REST APIs use HTTP methods like GET, POST, PUT, DELETE to communicate between systems.",
    "CSS flexbox simplifies the creation of responsive web layouts.",
    "Authentication vs authorization: who you are vs what you're allowed to do.",
    "SQL joins combine rows from two or more tables based on a related column.",
    # Data Science
    "Box plots show distribution, median, quartiles, and outliers of numerical data.",
    "Correlation does not imply causation — two variables can move together by chance.",
    "Principal Component Analysis reduces high-dimensional data while preserving variance.",
    "A/B testing compares two versions of something to determine which performs better.",
    # Career
    "How to prepare for a software engineering behavioral interview using STAR method.",
    "Negotiating your salary: research market rates and let them make the first offer.",
    "Building a portfolio: contribute to open source and build projects that solve real problems.",
    "The difference between a junior and senior engineer is not years but impact and ownership.",
]

if SBERT_AVAILABLE:
    print("Encoding corpus (one-time cost)...")
    corpus_embeddings = model.encode(CORPUS, normalize_embeddings=True)
    print(f"Corpus encoded: {corpus_embeddings.shape}")
else:
    print("[SIMULATION] Building corpus embeddings...")
    DIM = 384
    centers = np.random.randn(5, DIM)
    corpus_embeddings = []
    for c in range(5):
        for _ in range(4):
            e = centers[c] + np.random.randn(DIM) * 0.05
            e /= np.linalg.norm(e)
            corpus_embeddings.append(e)
    corpus_embeddings = np.array(corpus_embeddings)
    print(f"Simulated shape: {corpus_embeddings.shape}")


def semantic_search(query, corpus, corpus_embs, top_k=3):
    """Encode query and find top_k closest corpus docs."""
    if SBERT_AVAILABLE:
        q_emb = model.encode([query], normalize_embeddings=True)
    else:
        q_emb = corpus_embs[0:1] + np.random.randn(1, corpus_embs.shape[1]) * 0.1
        q_emb /= np.linalg.norm(q_emb)

    scores = (corpus_embs @ q_emb.T).flatten()
    top_idx = np.argsort(scores)[::-1][:top_k]
    return [(scores[i], corpus[i]) for i in top_idx]


queries = [
    "how to avoid model from learning training data too well",
    "job interview tips for programmers",
    "web API communication between services",
]

for query in queries:
    print("\n" + "═" * 65)
    print(f"  QUERY: '{query}'")
    print("═" * 65)
    for rank, (score, doc) in enumerate(semantic_search(query, CORPUS, corpus_embeddings), 1):
        print(f"  #{rank} [{score:.3f}] {'▓' * int(score*35)}")
        print(f"         {doc[:70]}")

---
## 7. Clustering Text with K-Means <a id='7-clustering'></a>

### The Library Sorting Analogy

A librarian dumps 1000 unsorted books on the floor. Instead of reading every book, they look at each book's fingerprint (embedding) and group similar fingerprints together. **K-Means on embeddings = automatic topic discovery without any labels.**

In [ ]:
# ── K-Means clustering on corpus embeddings ───────────────────────────────────

n_clusters = 5
kmeans = KMeans(n_clusters=n_clusters, random_state=42, n_init=10)
cluster_labels = kmeans.fit_predict(corpus_embeddings)

clusters = defaultdict(list)
for i, lbl in enumerate(cluster_labels):
    clusters[lbl].append(CORPUS[i])

print("K-MEANS DISCOVERED TOPICS (no labels given!)")
print("=" * 65)
for cid in sorted(clusters):
    print(f"\nCLUSTER {cid + 1} ({len(clusters[cid])} docs):")
    for doc in clusters[cid]:
        print(f"  • {doc[:68]}")

# Elbow method plot
inertias = []
K_range = range(2, 10)
for k in K_range:
    km = KMeans(n_clusters=k, random_state=42, n_init=10)
    km.fit(corpus_embeddings)
    inertias.append(km.inertia_)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].plot(K_range, inertias, 'bo-', lw=2, ms=8)
axes[0].axvline(x=5, color='red', ls='--', alpha=0.7, label='Optimal K=5')
axes[0].set_xlabel('Number of Clusters (K)')
axes[0].set_ylabel('Inertia')
axes[0].set_title('Elbow Method — pick K where curve bends')
axes[0].legend(); axes[0].grid(True, alpha=0.3)

pca2 = PCA(n_components=2, random_state=42)
reduced2 = pca2.fit_transform(corpus_embeddings)
cmap = cm.get_cmap('tab10')
for cid in range(n_clusters):
    mask = cluster_labels == cid
    axes[1].scatter(reduced2[mask, 0], reduced2[mask, 1],
                   c=[cmap(cid)], s=80, alpha=0.8,
                   label=f'Cluster {cid+1}', edgecolors='white')

for i, (x, y) in enumerate(reduced2):
    axes[1].annotate(CORPUS[i][:18], (x, y), fontsize=5, alpha=0.6,
                    xytext=(3, 3), textcoords='offset points')

axes[1].set_title('Clusters in 2D (PCA)')
axes[1].legend(fontsize=9); axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

---
## 8. Bi-Encoder vs Cross-Encoder <a id='8-bi-vs-cross'></a>

### The Interview Screening Analogy

**Bi-Encoder (SBERT) = Resume screening:**  
Each resume gets a score card once. HR compares score cards quickly (dot product). Fast but misses nuances.

**Cross-Encoder = Actual interview:**  
Interviewer reads resume AND talks to candidate *together* — sees exact fit for this role. Much more accurate, much slower.

### Best Practice: Two-Stage Pipeline

```
1,000,000 docs
    ↓  Bi-encoder retrieval  (~5ms)
   Top 100 candidates
    ↓  Cross-encoder re-rank  (~500ms)
   Top 10 final results
```

Speed of bi-encoder + accuracy of cross-encoder.

In [ ]:
# ── Two-stage retrieval pipeline ──────────────────────────────────────────────

HAS_CROSS = False
if SBERT_AVAILABLE:
    try:
        from sentence_transformers import CrossEncoder
        cross_encoder = CrossEncoder('cross-encoder/ms-marco-MiniLM-L-6-v2')
        HAS_CROSS = True
        print("Cross-encoder loaded.")
    except Exception as e:
        print(f"Cross-encoder not available: {e}")


def two_stage_search(query, corpus, corpus_embs, bi_k=8, cross_k=3):
    """Stage 1: fast bi-encoder retrieval. Stage 2: precise cross-encoder re-rank."""
    # Stage 1
    if SBERT_AVAILABLE:
        q_emb = model.encode([query], normalize_embeddings=True)
    else:
        q_emb = corpus_embs[0:1] + np.random.randn(1, corpus_embs.shape[1]) * 0.1
        q_emb /= np.linalg.norm(q_emb)

    bi_scores = (corpus_embs @ q_emb.T).flatten()
    top_idx = np.argsort(bi_scores)[::-1][:bi_k]
    candidates = [(corpus[i], float(bi_scores[i])) for i in top_idx]

    print(f"Stage 1 — Bi-encoder retrieved top-{bi_k}:")
    for doc, sc in candidates[:3]:
        print(f"  [{sc:.3f}] {doc[:62]}")
    print(f"  ... ({bi_k-3} more)")

    # Stage 2
    if HAS_CROSS:
        pairs = [[query, doc] for doc, _ in candidates]
        cross_scores = cross_encoder.predict(pairs)
        reranked = sorted(zip(cross_scores, [d for d, _ in candidates]),
                          key=lambda x: x[0], reverse=True)[:cross_k]
        print(f"\nStage 2 — Cross-encoder re-ranked top-{cross_k}:")
        for sc, doc in reranked:
            print(f"  [{sc:.3f}] {doc[:65]}")
    else:
        print(f"\nStage 2 — Bi-encoder top-{cross_k} (cross-encoder not loaded):")
        for doc, sc in candidates[:cross_k]:
            print(f"  [{sc:.3f}] {doc[:65]}")


print("=" * 65)
print("TWO-STAGE RETRIEVAL DEMO")
print("=" * 65)
two_stage_search(
    query="techniques to prevent my AI model from overfitting",
    corpus=CORPUS,
    corpus_embs=corpus_embeddings
)

print("\n" + "─"*65)
print("PERFORMANCE CONTEXT:")
print("  Bi-encoder only:  ~5 ms for 1M docs (dot products, GPU)")
print("  Cross-encoder:    ~50 ms/pair → 50,000 s for 1M docs")
print("  Two-stage:        5 ms + 500 ms (cross on top 100) = ~505 ms")

---
## 9. Fine-Tuning SBERT on Your Own Data <a id='9-finetuning'></a>

### When to Fine-Tune

Pre-trained `all-MiniLM-L6-v2` works great for general English. Fine-tune when:
- Your domain has specialized vocabulary (medical, legal, code, finance)
- You have labeled sentence pairs (even 1,000 is enough)
- The pre-trained model's performance on your task is below acceptable

### Loss Functions

| Loss | Data Needed | Use Case |
|---|---|---|
| `CosineSimilarityLoss` | Pairs + float score 0–1 | Similarity scoring |
| `MultipleNegativesRankingLoss` | Only positive pairs | Retrieval (very efficient) |
| `TripletLoss` | Anchor + positive + negative | Ranking |
| `OnlineContrastiveLoss` | Binary similar/not-similar | Classification |

In [ ]:
# ── Fine-tuning SBERT — code structure (runs if library available) ────────────

# Training data: customer support sentence pairs with similarity scores
train_pairs = [
    ("How do I reset my password?",       "What is the process to change my login credentials?", 1.0),
    ("My order hasn't arrived yet.",       "I'm still waiting for my package to be delivered.",   1.0),
    ("Can I get a refund?",                "What is your return policy for purchases?",            0.9),
    ("How do I track my shipment?",        "Where can I check my delivery status?",                1.0),
    ("My account is locked.",              "I cannot access my profile.",                          0.9),
    ("How do I cancel my subscription?",  "What payment methods do you accept?",                  0.3),
    ("Do you ship internationally?",       "What countries do you deliver to?",                    0.95),
    ("How do I reset my password?",        "What are your store hours?",                           0.0),
    ("My product is broken.",             "Do you have a student discount?",                       0.0),
    ("I need help with my bill.",          "There's an issue with my invoice.",                    0.95),
    ("How long does shipping take?",       "What is the estimated delivery time?",                 1.0),
    ("I want to update my address.",       "How do I change my shipping location?",                0.95),
]

if SBERT_AVAILABLE:
    # 1. Create InputExample objects
    train_examples = [
        InputExample(texts=[s1, s2], label=score)
        for s1, s2, score in train_pairs
    ]

    # 2. DataLoader
    train_dataloader = DataLoader(train_examples, shuffle=True, batch_size=4)

    # 3. Model + loss
    ft_model = SentenceTransformer('all-MiniLM-L6-v2')
    train_loss = losses.CosineSimilarityLoss(ft_model)

    # 4. Evaluator
    eval_pairs = [
        ("How do I view my order history?", "Where can I see my past purchases?", 1.0),
        ("Is my data safe?",                "Do you offer overnight shipping?",    0.0),
        ("The app keeps crashing.",         "Your application is not working properly.", 0.9),
    ]
    evaluator = EmbeddingSimilarityEvaluator(
        sentences1=[p[0] for p in eval_pairs],
        sentences2=[p[1] for p in eval_pairs],
        scores=[p[2] for p in eval_pairs],
        name='support-eval'
    )

    # 5. Train
    print("Fine-tuning on customer support data...")
    ft_model.fit(
        train_objectives=[(train_dataloader, train_loss)],
        evaluator=evaluator,
        epochs=3,
        warmup_steps=10,
        output_path='./sbert-customer-support',
        show_progress_bar=True
    )
    print("\nFine-tuning complete! Model saved to ./sbert-customer-support")

    # Test
    for s1, s2 in [
        ("How do I return a product?",  "I want to send back my purchase."),
        ("How do I return a product?",  "What time do you open on weekends?"),
    ]:
        e1 = ft_model.encode([s1], normalize_embeddings=True)
        e2 = ft_model.encode([s2], normalize_embeddings=True)
        sim = float(np.dot(e1, e2.T))
        print(f"  [{sim:.3f}] '{s1[:38]}' ↔ '{s2[:38]}'")
else:
    print("[Code structure — install sentence-transformers to run]")
    print()
    print("# 1. Wrap training data")
    print("train_examples = [InputExample(texts=[s1, s2], label=score) for ...]")
    print()
    print("# 2. DataLoader")
    print("train_dataloader = DataLoader(train_examples, shuffle=True, batch_size=16)")
    print()
    print("# 3. Loss function")
    print("model = SentenceTransformer('all-MiniLM-L6-v2')")
    print("train_loss = losses.CosineSimilarityLoss(model)")
    print()
    print("# 4. Fine-tune")
    print("model.fit(train_objectives=[(train_dataloader, train_loss)],")
    print("          epochs=3, warmup_steps=100)")
    print()
    print(f"Training pairs preview:")
    for s1, s2, sc in train_pairs[:3]:
        print(f"  [{sc:.1f}] '{s1[:38]}' ↔ '{s2[:38]}'")

---
## 10. Mini Project — FAQ Semantic Search Engine <a id='10-mini-project'></a>

### What We're Building

A smart FAQ bot for an e-commerce company. Users type questions in any phrasing — the bot finds the best FAQ answer.

**Traditional approach:** keyword matching → fails when user says "parcel" instead of "package"  
**Our approach:** SBERT → works regardless of phrasing, with confidence thresholds and fallback to human support

In [ ]:
# ── FAQ Database ──────────────────────────────────────────────────────────────

FAQ_DATABASE = [
    {'q': 'How long does standard shipping take?',
     'a': 'Standard shipping takes 5-7 business days. Express (2-day) is available at checkout.'},
    {'q': 'What is your return policy?',
     'a': 'We offer 30-day hassle-free returns. Items must be unused in original packaging.'},
    {'q': 'How do I track my order?',
     'a': 'Once shipped, you receive a tracking number by email. Track at shopease.com/track.'},
    {'q': 'Do you offer international shipping?',
     'a': 'Yes — we ship to 45+ countries. Delivery takes 10-21 days; import duties may apply.'},
    {'q': 'How do I cancel my order?',
     'a': 'Orders can be cancelled within 1 hour. After that, you will need to return the item.'},
    {'q': 'What payment methods do you accept?',
     'a': 'We accept Visa, Mastercard, AmEx, PayPal, Apple Pay, Google Pay, and store credit.'},
    {'q': 'How do I change my delivery address?',
     'a': 'Go to Orders > Select Order > Edit Address before your order ships.'},
    {'q': 'Is my personal information secure?',
     'a': 'Yes — 256-bit SSL encryption. We never sell your data to third parties.'},
    {'q': 'How do I apply a discount coupon?',
     'a': 'Enter your code in the Promo Code box at checkout and click Apply.'},
    {'q': 'What should I do if I received a damaged item?',
     'a': 'Photo the damage and contact us within 48h. We will send a replacement free of charge.'},
    {'q': 'How do I create an account?',
     'a': 'Click Sign Up on the homepage. Register with email or sign in via Google/Facebook.'},
    {'q': 'Do you have a loyalty rewards program?',
     'a': 'ShopEasy Rewards: 1 point per dollar spent. 100 points = $5 off. Points never expire.'},
]

faq_questions = [item['q'] for item in FAQ_DATABASE]
faq_answers   = [item['a'] for item in FAQ_DATABASE]

if SBERT_AVAILABLE:
    search_model = SentenceTransformer('all-MiniLM-L6-v2')
    faq_embeddings = search_model.encode(faq_questions, normalize_embeddings=True)
    print(f"FAQ index built: {len(FAQ_DATABASE)} questions encoded | shape: {faq_embeddings.shape}")
else:
    print("[SIMULATION] Building FAQ index...")
    DIM = 384
    # 4 rough topic groups: shipping, returns, payment, account
    groups = [[0, 2, 3, 4, 6], [1, 9], [5, 8], [7, 10, 11]]
    centers = np.random.randn(len(groups), DIM)
    faq_embeddings = np.zeros((len(FAQ_DATABASE), DIM))
    for g_idx, grp in enumerate(groups):
        for faq_idx in grp:
            e = centers[g_idx] + np.random.randn(DIM) * 0.05
            faq_embeddings[faq_idx] = e
    faq_embeddings /= np.linalg.norm(faq_embeddings, axis=1, keepdims=True)
    print(f"Simulated FAQ embeddings: {faq_embeddings.shape}")

In [ ]:
# ── FAQ Search Engine ─────────────────────────────────────────────────────────

class FAQSearchEngine:
    """
    Semantic FAQ bot.
    - Encodes all FAQ questions at startup (one-time)
    - At query time: encode query → nearest FAQ → return answer
    - Falls back to human support if confidence < threshold
    """

    def __init__(self, faq_data, faq_embs, threshold=0.3):
        self.faq_data = faq_data
        self.faq_embs = faq_embs
        self.threshold = threshold
        self.fallback = ("I couldn't find a good match. Please contact "
                         "support@shopease.com or call 1-800-SHOP-EASY.")
        self.total = self.escalated = 0

    def search(self, query, top_k=2):
        self.total += 1
        t0 = time.time()

        if SBERT_AVAILABLE:
            q_emb = search_model.encode([query], normalize_embeddings=True)
        else:
            q_emb = self.faq_embs[0:1] + np.random.randn(1, self.faq_embs.shape[1]) * 0.1
            q_emb /= np.linalg.norm(q_emb)

        scores = (self.faq_embs @ q_emb.T).flatten()
        top_idx = np.argsort(scores)[::-1][:top_k]
        best, best_score = top_idx[0], scores[top_idx[0]]
        ms = (time.time() - t0) * 1000

        print(f"\n{'─'*60}")
        print(f"USER:       {query}")
        print(f"Confidence: {best_score:.3f} | Time: {ms:.1f}ms")

        if best_score < self.threshold:
            self.escalated += 1
            print(f"[LOW CONFIDENCE → escalating to human support]")
            print(f"ANSWER: {self.fallback}")
        else:
            print(f"MATCHED: '{self.faq_data[best]['q']}'")
            print(f"ANSWER:  {self.faq_data[best]['a']}")
            if top_k > 1:
                alt = top_idx[1]
                print(f"Also relevant: '{self.faq_data[alt]['q']}' [{scores[alt]:.2f}]")

    def stats(self):
        answered = self.total - self.escalated
        print(f"\n{'═'*45}")
        print(f"Total queries:  {self.total}")
        print(f"Self-served:    {answered} ({answered/max(1,self.total)*100:.0f}%)")
        print(f"Escalated:      {self.escalated} ({self.escalated/max(1,self.total)*100:.0f}%)")


bot = FAQSearchEngine(FAQ_DATABASE, faq_embeddings, threshold=0.25)

test_queries = [
    "When will my parcel arrive?",
    "I got the wrong product, what do I do?",
    "How can I use my promo code?",
    "Is it possible to get my money back?",
    "What credit cards are accepted?",
    "Tell me about the weather forecast.",   # out-of-scope → fallback
]

print("ShopEasy FAQ Bot Demo")
print("=" * 60)
for q in test_queries:
    bot.search(q)

bot.stats()

In [ ]:
# ── Visualize FAQ embedding space ─────────────────────────────────────────────

faq_pca = PCA(n_components=2, random_state=42)
faq_2d = faq_pca.fit_transform(faq_embeddings)

faq_topics = ['Shipping','Returns','Shipping','Shipping','Orders',
              'Payment','Shipping','Security','Payment','Returns',
              'Account','Rewards']
topic_colors = {'Shipping':'#3498db','Returns':'#e74c3c','Orders':'#9b59b6',
                'Payment':'#2ecc71','Security':'#f39c12','Account':'#1abc9c',
                'Rewards':'#e67e22'}

fig, ax = plt.subplots(figsize=(11, 7))
seen = set()
for i, (x, y) in enumerate(faq_2d):
    t = faq_topics[i]
    ax.scatter(x, y, c=topic_colors[t], s=150, alpha=0.85,
               label=t if t not in seen else '', edgecolors='white', lw=1.5)
    seen.add(t)
    ax.annotate(faq_questions[i][:32], (x, y), fontsize=7, alpha=0.85,
                xytext=(5, 5), textcoords='offset points')

ax.set_title('FAQ Questions in Embedding Space\nSimilar topics cluster — enables fast semantic routing', fontsize=12)
ax.legend(loc='upper right', fontsize=9)
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

print("Business impact:")
print("  • Self-service rate: ~80-90% of queries answered automatically")
print("  • Works for any phrasing — not just exact keyword matches")
print("  • Confidence threshold routes unclear queries to human agents")
print("  • Same architecture scales to 100,000+ FAQ items with FAISS")

---
## 11. Common Pitfalls <a id='11-pitfalls'></a>

In [ ]:
print("""
╔══════════════════════════════════════════════════════════════════╗
║          SENTENCE TRANSFORMERS — COMMON PITFALLS                 ║
╠══════════════════════════════════════════════════════════════════╣
║                                                                  ║
║  1. WRONG: Using dot product without normalizing                 ║
║     emb = model.encode(texts)           ← not normalized         ║
║     np.dot(emb[0], emb[1])              ← this is NOT cos sim!   ║
║  RIGHT:                                                          ║
║     emb = model.encode(texts, normalize_embeddings=True)         ║
║     np.dot(emb[0], emb[1])              ← NOW it's cosine sim ✓  ║
║                                                                  ║
║  2. WRONG: Re-encoding corpus on every query                     ║
║     for q in queries:                                            ║
║         corpus_embs = model.encode(corpus)  ← repeated, slow!   ║
║  RIGHT: Encode corpus ONCE, cache to disk                        ║
║     corpus_embs = model.encode(corpus)                           ║
║     np.save('corpus_embs.npy', corpus_embs)  # reuse next time  ║
║                                                                  ║
║  3. WRONG: Using raw BERT for sentence similarity                ║
║     model = AutoModel.from_pretrained('bert-base-uncased')       ║
║     # BERT [CLS] is NOT a sentence embedding — it's trained      ║
║     # for MLM/NSP, not similarity!                               ║
║  RIGHT:                                                          ║
║     model = SentenceTransformer('all-MiniLM-L6-v2')  ✓          ║
║                                                                  ║
║  4. WRONG: Comparing embeddings from different models            ║
║     emb1 = model_a.encode("Hello")                               ║
║     emb2 = model_b.encode("Hello")                               ║
║     cosine_similarity(emb1, emb2)  ← meaningless result          ║
║  Always use the SAME model for all texts you compare.            ║
║                                                                  ║
║  5. WRONG: Trusting high similarity for negation/antonyms        ║
║     "I love cats" vs "I hate cats" → both about cats → ~similar  ║
║     SBERT is NOT great at detecting negation.                    ║
║  Use a cross-encoder for high-stakes semantic comparison.        ║
║                                                                  ║
║  6. WRONG: Feeding very long documents                           ║
║     model.encode(long_5000_word_doc)  ← truncated at 256 tokens! ║
║  RIGHT: Chunk long docs and average their embeddings,            ║
║  or use longformer-based SBERT models.                           ║
║                                                                  ║
║  7. WRONG: No confidence threshold                               ║
║     Always returning a result even when similarity = 0.1         ║
║     → Misleading answers for out-of-scope queries                ║
║  RIGHT: Set a minimum confidence threshold; fallback to human.   ║
║                                                                  ║
╚══════════════════════════════════════════════════════════════════╝
""")

---
## 12. Interview Q&A <a id='12-interview'></a>

In [ ]:
qa = [
    ("What are sentence embeddings and why are they useful?",
     "Dense vector representations of entire sentences where semantically similar sentences "
     "map to nearby vectors. Unlike TF-IDF/bag-of-words, they capture meaning not exact words. "
     "Computed once per sentence, then compared via fast dot products — enabling semantic search, "
     "clustering, duplicate detection, and similarity scoring at scale."),

    ("How does SBERT differ from raw BERT?",
     "Raw BERT requires BOTH sentences together (cross-encoder), making N² comparisons. "
     "BERT's [CLS] token is also a poor sentence representation (trained for MLM/NSP, not similarity). "
     "SBERT uses a Siamese network trained on NLI/STS with contrastive/triplet loss — "
     "each sentence encoded independently via mean pooling. Reduces 10,000-sentence comparison "
     "from 65 hours (BERT) to 5 seconds."),

    ("What is the difference between bi-encoder and cross-encoder?",
     "Bi-encoder: encodes each sentence independently → compare with dot product. "
     "O(1) comparison after encoding. Fast but misses fine-grained token interactions. "
     "Cross-encoder: passes both sentences together → sees all token interactions. "
     "More accurate but O(N) per query. Best practice: bi-encoder retrieves top-K, "
     "cross-encoder re-ranks those K results."),

    ("What is cosine similarity and why prefer it over Euclidean distance for embeddings?",
     "Cosine similarity = cos(angle between vectors) = (A·B)/(|A||B|). Range [-1, 1]. "
     "Preferred because it's scale-invariant — a long and short paraphrase of the same text "
     "shouldn't be far apart just due to magnitude. When embeddings are L2-normalized, "
     "cosine sim = dot product → GPU-friendly matrix multiply."),

    ("When would you fine-tune SBERT?",
     "When: (1) domain has specialized vocabulary (medical, legal, code), "
     "(2) pre-trained model underperforms on your benchmark, "
     "(3) you have labeled pairs (even 1000+ is sufficient). "
     "Use CosineSimilarityLoss for scored pairs, MultipleNegativesRankingLoss when only "
     "positive pairs available (very data-efficient for retrieval), "
     "TripletLoss for anchor/positive/negative triples."),

    ("How would you scale semantic search to 100 million documents?",
     "Brute-force cosine (O(N·d)) is infeasible at 100M. Use Approximate Nearest Neighbor (ANN): "
     "FAISS (IVF index — partition space, search only relevant partitions, 100× speedup), "
     "Annoy (tree-based), ScaNN (Google, state-of-the-art). "
     "Managed options: Pinecone, Qdrant, Weaviate, Milvus. "
     "Typical pipeline: SBERT bi-encoder → FAISS top-1000 → cross-encoder re-rank top-10."),

    ("What is mean pooling and why is it better than using the [CLS] token?",
     "Mean pooling averages all token embeddings weighted by the attention mask (ignores [PAD]). "
     "The [CLS] token was designed for BERT's original pre-training tasks (MLM + NSP), "
     "not sentence representation. Empirically, mean pooling incorporates information from ALL "
     "tokens, giving a more complete semantic representation."),
]

print("=" * 70)
print("  INTERVIEW Q&A — SENTENCE TRANSFORMERS")
print("=" * 70)
for i, (q, a) in enumerate(qa, 1):
    print(f"\nQ{i}: {q}")
    print(f"\nA{i}: {a}")
    print("\n" + "─" * 70)

---
## 13. Resources <a id='13-resources'></a>

### Official Docs
- **Sentence Transformers**: https://www.sbert.net/
- **Pre-trained models**: https://www.sbert.net/docs/pretrained_models.html
- **Training examples**: https://github.com/UKPLab/sentence-transformers/tree/master/examples

### Research Papers
- **SBERT** (Reimers & Gurevych, 2019): https://arxiv.org/abs/1908.10084
- **SimCSE** (contrastive sentence embeddings): https://arxiv.org/abs/2104.08821
- **DPR** (Dense Passage Retrieval): https://arxiv.org/abs/2004.04906

### Video Tutorials
- **Sentence Transformers explained** (James Briggs): https://youtu.be/OATCgQtNX2o
- **Semantic search from scratch**: https://youtu.be/H7TsE1MFnRQ
- **Vector embeddings crash course**: https://youtu.be/ySus5ZS0b94

### Vector Databases (for scaling)
- **FAISS**: https://github.com/facebookresearch/faiss
- **Pinecone**: https://www.pinecone.io/
- **Qdrant**: https://qdrant.tech/
- **Chroma**: https://www.trychroma.com/

### Datasets for Fine-Tuning
- **Quora Question Pairs**: https://huggingface.co/datasets/quora
- **SNLI / MultiNLI**: https://huggingface.co/datasets/snli
- **MS MARCO**: https://huggingface.co/datasets/ms_marco

---
## 14. Summary & What's Next <a id='14-summary'></a>

### What You Learned

| Concept | Key Takeaway |
|---|---|
| Sentence embeddings | Fixed-size vectors encoding the *meaning* of full sentences |
| SBERT architecture | Siamese BERT + mean pooling + contrastive training |
| Cosine similarity | Angle between vectors = semantic distance; normalize for fast dot product |
| Semantic search | Pre-compute corpus embeddings once; query = dot product |
| K-Means clustering | Automatic topic discovery — no labels needed |
| Bi-encoder vs Cross-encoder | Speed vs accuracy; combine in two-stage pipeline |
| Fine-tuning | `InputExample` + `losses.*` to adapt to your domain in <5 epochs |
| FAQ engine | Confidence threshold + fallback = production-ready bot |

### Golden Rule

**Start with `all-MiniLM-L6-v2`.** Fast, good quality, works for 80% of use cases. Upgrade to `all-mpnet-base-v2` only when benchmarks demand it.

### What's Next — Computer Vision

| Notebook | Topic |
|---|---|
| `OpenCV` | Image processing, filters, edge detection, video analysis |
| `Torchvision` | CNN training, transfer learning, image augmentation |
| `YOLO_Ultralytics` | Real-time object detection in images and video |
| `Detectron2` | Instance segmentation and keypoint detection |

**You now have the full NLP stack — word vectors → transformers → sentence embeddings. Time for Computer Vision!**